In [1]:
import os
os.chdir("../")

In [2]:
%pwd

'/Users/prasad/Learning/Projects/Text_Summarization_Project'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    data_path: Path
    model_ckpt: Path
    num_train_epochs = int
    warmup_steps = int
    per_device_train_batch_size = int
    per_device_eval_batch_size = int
    weight_decay = float
    logging_steps = int
    eval_strategy = str
    eval_steps = int
    save_steps = float
    gradient_accumulation_steps = int

In [4]:
from textSummarization.constants import *
from textSummarization.utils.common import read_yaml, create_directories


class ConfigurationManager:
    def __init__(
        self,
        config_file_path: Path = CONFIG_FILE_PATH,
        params_file_path: Path = PARAMS_FILE_PATH
    ) -> None:
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
    
        create_directories([self.config.artificats_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        model_trainer_config = self.config.model_trainer
        model_trainer_params = self.params.TrainingArguments

        create_directories([model_trainer_config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=model_trainer_config.root_dir,
            data_path=model_trainer_config.data_path,
            model_ckpt=model_trainer_config.model_ckpt,
            num_train_epochs=model_trainer_params.num_train_epochs,
            warmup_steps=model_trainer_params.warmup_steps,
            per_device_train_batch_size=model_trainer_params.per_device_train_batch_size,
            per_device_eval_batch_size=model_trainer_params.per_device_eval_batch_size,
            weight_decay=model_trainer_params.weight_decay,
            logging_steps=model_trainer_params.logging_steps,
            eval_strategy=model_trainer_params.eval_strategy,
            eval_steps=model_trainer_params.eval_steps,
            save_steps=model_trainer_params.save_steps,
            gradient_accumulation_steps=model_trainer_params.gradient_accumulation_steps
        )

        return model_trainer_config

In [5]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorForSeq2Seq
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import torch

In [ ]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig) -> None:
        self.config = config

    def train(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_ckpt)
        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_ckpt).to(device)
        seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

        dataset_samsum_pt = load_from_disk(self.config.data_path)
        training_args = TrainingArguments(
            output_dir=self.config.root_dir,
            num_train_epochs=self.config.num_train_epochs,
            warmup_steps=self.config.warmup_steps,
            per_device_train_batch_size=self.config.per_device_train_batch_size,
            per_device_eval_batch_size=self.config.per_device_eval_batch_size,
            weight_decay=self.config.weight_decay,
            logging_steps=self.config.logging_steps,
            eval_strategy=self.config.eval_strategy,
            eval_steps=self.config.eval_steps,
            save_steps=self.config.save_steps,
            gradient_accumulation_steps=self.config.gradient_accumulation_steps
        )

        trainer = Trainer(
            model=model_pegasus,
            args=training_args,
            tokenizer=tokenizer,
            data_collator=seq2seq_data_collator,
            train_dataset=dataset_samsum_pt["test"], # TODO : change to train
            eval_dataset=dataset_samsum_pt["validation"]
        )

        trainer.train()

        model_pegasus.save_pretrained(os.path.join(self.config.root_dir, "pegasus-samsum-model"))

        tokenizer.save_pretrained(os.path.join(self.config.root_dir, "pegasus-samsum-tokenizer"))

In [ ]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer = ModelTrainer(config=model_trainer_config)
    model_trainer.train()
except Exception as e:
    logger.info(f"pipeline exception : {e}")
    raise e